In [8]:
def pois_per_hex (poi_list, hex_data, poi_dataset="urban-access-bot.bot_pois"):
    """
    Генерирует SQL-запрос для подсчёта POI по гексагонам.

    Параметры:
    poi_list: cписок названий таблиц POI
    hex_data: название таблицы с гексагонами
    poi_dataset: название датасета с таблицами POI
    
    """
    # Объединяем все таблицы POI, добавляя столбец category
    union_parts = []
    for poi in poi_list:
        union_parts.append(f"""
  SELECT
    '{poi}' AS category, geometry
  FROM `{poi_dataset}.{poi}`""")
    
    union_all = "\n  UNION ALL\n".join(union_parts)

    # Формируем список столбцов для каждой категории с количеством объектов в гексагоне
    countif_columns = []
    for poi in poi_list:
        countif_columns.append(f"  COUNTIF(poi.category = '{poi}') AS `{poi.upper()}_COUNT`")
    
    countif_cols = ",\n".join(countif_columns)

    # Итоговый запрос
    query = f"""
WITH all_pois AS (
  {union_all}
)
SELECT
  hex.index AS H3_id,
  ANY_VALUE(hex.geometry) AS geometry,
{countif_cols}
FROM `{hex_data}` AS hex
JOIN all_pois AS poi
  ON ST_INTERSECTS(hex.geometry, poi.geometry)
GROUP BY hex.index
ORDER BY hex.index;
"""
    return query

if __name__ == "__main__":
    poi_list = [
        'bot_rests_osm_tl',
        'clinics_osm_me',
        'hospitals_osm_me',
        'kindergarten_osm_dt',
        'malls_osm_me',
        'outpost_osm_dt',
        'pharmacies_osm_kl',
        'police_osm_dt',
        'post_offices_osm_kl',
        'schools_osm_me',
        'shops_osm_kl',
        'sport_osm_dt'
    ]

    hex_data = 'urban-access-bot.bot_geo.h3_10res_london_kl'

    sql_script = pois_per_hex (poi_list, hex_data)
    print(sql_script)


WITH all_pois AS (
  
  SELECT
    'bot_rests_osm_tl' AS category, geometry
  FROM `urban-access-bot.bot_pois.bot_rests_osm_tl`
  UNION ALL

  SELECT
    'clinics_osm_me' AS category, geometry
  FROM `urban-access-bot.bot_pois.clinics_osm_me`
  UNION ALL

  SELECT
    'hospitals_osm_me' AS category, geometry
  FROM `urban-access-bot.bot_pois.hospitals_osm_me`
  UNION ALL

  SELECT
    'kindergarten_osm_dt' AS category, geometry
  FROM `urban-access-bot.bot_pois.kindergarten_osm_dt`
  UNION ALL

  SELECT
    'malls_osm_me' AS category, geometry
  FROM `urban-access-bot.bot_pois.malls_osm_me`
  UNION ALL

  SELECT
    'outpost_osm_dt' AS category, geometry
  FROM `urban-access-bot.bot_pois.outpost_osm_dt`
  UNION ALL

  SELECT
    'pharmacies_osm_kl' AS category, geometry
  FROM `urban-access-bot.bot_pois.pharmacies_osm_kl`
  UNION ALL

  SELECT
    'police_osm_dt' AS category, geometry
  FROM `urban-access-bot.bot_pois.police_osm_dt`
  UNION ALL

  SELECT
    'post_offices_osm_kl' AS c